# Data Ingestion Pipeline

In [ ]:
# Imports
import os
import requests

In [2]:
# Current Working directory
%pwd

'd:\\Projects\\DataScience_Project\\research'

In [3]:
# Change working directory
os.chdir("../")       # Goes one main project directory by moving one folder outside
%pwd

'd:\\Projects\\DataScience_Project'

In [ ]:
# Data Ingestion Config
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataIngestionConfig:
    root_dir : Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [5]:
# Import all defined constants
from src.datascience.constants import *

# Import read yaml and create directory functions
from src.datascience.utils.common import read_yaml, create_directory


class ConfigurationManager:
    # Constructor
    def __init__(self, 
                 config_filepath = CONFIG_FILE_PATH, 
                 params_filepath = PARAMS_FILE_PATH, 
                 schema_filepath = SCHEMA_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        # Create artifacts root directory
        create_directory([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directory([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir = config.root_dir,
            source_URL = config.source_URL,
            local_data_file = config.local_data_file,
            unzip_dir = config.unzip_dir 
        )

        return data_ingestion_config


In [6]:
# Component - Data Ingestion

import urllib.request as request
from src.datascience import logger
import zipfile

class DataIngestion:
    def __init__(self, config:DataIngestionConfig):
        self.config = config
    
    # Download Zip file
    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url = self.config.source_URL,
                filename = self.config.local_data_file
            )
            logger.info(f"{filename} downloaded with following info: \n{headers}")
        else:
            logger.info(f"File already exists")
    
    # Extract Zip file
    def extract_zip(self):
        """
        Extracts the zip file into the data directory
        Function returns none
        """

        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as file:
            file.extractall(unzip_path) 

In [ ]:
# Run Pipeline

try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config= data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip()
except Exception as e:
    raise e